# Cindochick — Generate Dataset + Latih LoRA (Kaggle GPU gratis)

## Cara pakai (5 menit setup)
1. Upload file ini ke Kaggle: Code → New Notebook → titik tiga → Upload.
2. Kanan atas: Accelerator = **GPU T4 x2**, Internet = **ON**.
3. Akun Kaggle harus sudah verifikasi nomor HP (gratis, sekali saja) agar GPU nyala.
4. **Run All**. Bagian 1 generate ±20 foto (±30 mnt). Bagian 2 latih LoRA (±1–2 jam).
5. Download hasil dari panel Output (kanan): `dataset.zip` dan nanti `cindochick-lora.zip`.

Total ±2–3 jam dari kuota gratis 30 jam/minggu. Masih sisa banyak.

## Bagian 0 — Cek GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.device_count())

## Bagian 1 — Install + download model

Default: **Pony Diffusion V6 XL** (paling cocok untuk karakter pinup). Kalau gagal download, ganti `MODEL_ID` ke `stabilityai/stable-diffusion-xl-base-1.0`.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors peft huggingface_hub

MODEL_ID = 'AstraliteHeart/pony-diffusion-v6-xl'  # fallback: 'stabilityai/stable-diffusion-xl-base-1.0'
TRIGGER = 'cindochick'

import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to('cuda')
pipe.enable_model_cpu_offload()
print('Model siap:', MODEL_ID)

## Bagian 2 — Generate 20 foto dataset

Aturan yang dipakai di sini (standar LoRA karakter):
- **Blok identitas SAMA PERSIS** di semua prompt → inilah yang mengunci muka.
- 70% wajah jelas menghadap kamera, 30% variasi angle.
- Tiap foto disimpan + file `.txt` berisi trigger + deskripsi (caption untuk latih).

In [ ]:
import os
from pathlib import Path

IDENTITY = (
    'cindochick, chinese-indonesian young adult woman, long straight black hair, '
    'warm tan skin, soft round face, big brown eyes, natural makeup, '
    'curvy thick body, long legs, '
    'score_9, score_8_up, score_7_up, photorealistic, 35mm photo'
)
NEG = 'score_6, score_5, score_4, blurry, deformed, cartoon, anime, watermark, text, logo'

SCENES = [
    'fitted white tank top and denim shorts, standing in bedroom, window light, smiling at camera',
    'black spaghetti-strap dress, sitting on cafe chair, night cafe bokeh, soft smile',
    'oversized grey hoodie slipping off one shoulder, messy bun hair, cozy room, selfie angle',
    'red summer dress, standing on jakarta street at dusk, city lights, looking back at camera',
    'white crop top and high-waist jeans, leaning on balcony railing, daylight, laughing',
    'silk black robe over camisole, hotel room lamp light, portrait, calm expression',
    'yellow sundress, beach bali background, wind in hair, walking pose',
    'gym outfit, sports bra and leggings, gym mirror selfie, ponytail',
    'pink satin pajama set, sitting on bed with laptop, warm lamp, winking',
    'denim jacket over white tee, rooftop sunset, arms crossed, confident',
    'floral blouse and skirt, coffee shop interior, holding latte, candid smile',
    'black evening gown, hotel lobby staircase, elegant pose, side profile then face to camera',
    'casual oversized t-shirt, convenience store night, neon sign glow, playful pout',
    'traditional kebaya modern red, studio portrait, soft box light, graceful',
    'swimsuit cover-up sarong at pool edge, sunglasses on head, resort bali',
    'office blouse and pencil skirt, office window, professional smile',
    'hoodie and shorts, gaming room with LED lights, holding controller, excited',
    'white shirt wet-look rain shoot, city night rain, dramatic lighting',
    'batik modern dress, museum interior, soft daylight, gentle smile',
    'close-up face portrait, freckles visible, golden hour sunlight, ultra detailed skin',
]

OUT = Path('/kaggle/working/dataset')
OUT.mkdir(parents=True, exist_ok=True)

for i, scene in enumerate(SCENES, start=1):
    prompt = f'{IDENTITY}, {scene}'
    img = pipe(prompt=prompt, negative_prompt=NEG, height=1024, width=768,
               num_inference_steps=30, guidance_scale=6.5,
               generator=torch.Generator('cuda').manual_seed(1000 + i)).images[0]
    img.save(OUT / f'cindochick_{i:02d}.png')
    (OUT / f'cindochick_{i:02d}.txt').write_text(f'{TRIGGER}, {scene}')
    print(f'{i}/20 jadi: cindochick_{i:02d}.png')

print('SELESAI:', len(list(OUT.glob("*.png"))), 'foto')

## Bagian 3 — Cek + download dataset

In [ ]:
from pathlib import Path
files = sorted(Path('/kaggle/working/dataset').glob('*.png'))
print(len(files), 'foto.')
# tampilkan 4 contoh acak buat cek muka konsisten
from PIL import Image
import random
sample = random.sample(files, min(4, len(files)))
grid = Image.new('RGB', (768 * 2, 1024 * 2))
for k, f in enumerate(sample):
    grid.paste(Image.open(f).resize((768, 1024)), ((k % 2) * 768, (k // 2) * 1024))
grid.resize((768, 1024))
!mkdir -p /kaggle/working/output
grid.save('/kaggle/working/output/preview.jpg')
grid

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/output/dataset.zip dataset
print('dataset.zip siap di panel Output kanan')

## Bagian 4 — Latih LoRA (Ostris AI-Toolkit)

Jalankan 2 cell di bawah **setelah dataset jadi**. Latih ±1–2 jam (500 steps). Output: `cindochick-lora.zip` berisi file `.safetensors` — inilah 'muka permanen' karaktermu, bisa dipakai di generate mana pun.

In [ ]:
!git clone -q https://github.com/ostris/ai-toolkit.git /kaggle/working/ai-toolkit
!pip install -q -r /kaggle/working/ai-toolkit/requirements.txt

yaml = '''
job: extension
config:
  name: "cindochick"
  process:
    - type: 'sd_trainer'
      training_folder: "/kaggle/working/output"
      device: cuda:0
      trigger_word: "cindochick"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 100
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/kaggle/working/dataset"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [768, 1024]
      train:
        batch_size: 1
        steps: 500
        ema_config:
          use_ema: true
          ema_decay: 0.999
        dtype: bf16
        lr: 1e-4
        optimizer: "adamw8bit"
        optimizer_params:
          weight_decay: 0.01
      model:
        name_or_path: "AstraliteHeart/pony-diffusion-v6-xl"
        quantize: true
      sample:
        sampler: "flowmatch"
        sample_every: 100
        width: 768
        height: 1024
        prompts:
          - "cindochick, white tank top and jeans, cafe, smiling at camera"
        neg: "blurry, deformed, cartoon, watermark"
        seed: 42
        walk_seed: true
        guidance_scale: 6.5
        sample_steps: 25
'''
open('/kaggle/working/ai-toolkit/cindochick.yaml', 'w').write(yaml)
print('konfig latih siap')

In [ ]:
# JALANKAN INI TERAKHIR — tunggu sampai selesai (±1-2 jam), jangan tutup tab
!cd /kaggle/working/ai-toolkit && python run.py cindochick.yaml

In [ ]:
!cd /kaggle/working/output && zip -qr cindochick-lora.zip cindochick 2>/dev/null; ls -la /kaggle/working/output/
print('Ambil cindochick-lora.zip dari panel Output kanan')